In [ ]:
#import des données :
%pip install -r requirements.txt
import pandas as pd
import matplotlib.pyplot as plt
from cartiflette import carti_download
import seaborn as sns
import geopandas as gpd

url = "https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb"
df = pd.read_csv(
    url, 
    dtype={
        'code_departement': str, 
        'code_commune': str } )


In [ ]:
#Question4)
#On regarde la table, ses valeurs manquantes et le type des colonnes
print(df.dtypes)
print(df.isnull().sum())
#On ne laisse que les votes exprimés
exclusions = ['abstentions', 'blancs', 'nuls']
df_exprimes = df[~df['nom'].isin(exclusions)].copy()
#On crée candidat
df_exprimes['candidat'] = df_exprimes['prenom'] + ' ' + df_exprimes['nom']
#On regroupe
score_departements = df_exprimes.groupby(['code_departement', 'candidat']).agg({
    'voix': 'sum'
}).reset_index()
#On rajoute les %
total_par_dept = score_departements.groupby('code_departement')['voix'].transform('sum')
score_departements['score'] = (score_departements['voix'] / total_par_dept) * 100

In [ ]:
#Vérification TABLE 2

verif_table2=score_departements[score_departements['code_departement'] == '11'].sort_values(by='voix', ascending=False)
print(verif_table2)

In [13]:
#Question5
#On crée les stats nationales
total_votes_national = score_departements['voix'].sum()
stats_national = score_departements.groupby('candidat').agg({
    'voix': 'sum'
}).reset_index()

stats_national.columns = ['candidat', 'votes_national']

stats_national['score_national'] = (stats_national['votes_national'] / total_votes_national) * 100
score_departements = score_departements.rename(columns={
    'voix': 'votes_departement',
    'score': 'score_departement'
})

#On fusionne
score_departements = pd.merge(score_departements, stats_national, on='candidat')



In [ ]:
#Vérification table3
verif_table_3 = score_departements[score_departements['code_departement'] == '11'].sort_values(by='votes_departement', ascending=False)
print(verif_table_3)

In [ ]:
# Question 6 

# Calcul de la surreprésentation relative (en %)
score_departements['surrepresentation'] = (
    (score_departements['score_departement'] / score_departements['score_national']) - 1
) * 100

verif_q6 = score_departements[
    (score_departements['code_departement'] == '11') &
    (score_departements['candidat'].str.contains('LE PEN'))
][['candidat', 'score_departement', 'score_national', 'surrepresentation']]

print(verif_q6)
# On obtient bien un score d'environ 30% pour Marine Le Pen comme dans la table 3 du sujet. 


In [ ]:
#Question 7 

def plot_top_5(df, nom_candidat):
    data = (
        df[df['candidat'].str.contains(nom_candidat, regex=False)]
        .assign(abs_surr=lambda x: x['surrepresentation'].abs())
        .nlargest(5, 'abs_surr')
        .sort_values('surrepresentation')  
    )
    plt.figure(figsize=(8, 5))
    colors = ['steelblue' if v < 0 else 'steelblue' for v in data['surrepresentation']]
    sns.barplot(x='surrepresentation', y='code_departement', data=data, color='steelblue')
    plt.title(f"Top 5 des surreprésentations de {nom_candidat}")
    plt.xlabel("Surreprésentation")
    plt.ylabel("Département")
    plt.axvline(0, color='black', linewidth=0.8)
    plt.tight_layout()
    plt.show()

plot_top_5(score_departements, "ZEMMOUR")


In [ ]:
#Question 8 

departement_borders = carti_download(
    values=["France"],
    crs=4326,
    borders="DEPARTEMENT",
    vectorfile_format="geojson",
    simplification=50,
    filter_by="FRANCE_ENTIERE_DROM_RAPPROCHES",
    source="EXPRESS-COG-CARTO-TERRITOIRE",
    year=2022
)
def tracer_carte_candidat(nom_candidat):
    df_visu = score_departements[
        score_departements['candidat'].str.contains(nom_candidat, regex=False)
    ].copy()
    
    carte_data = departement_borders.merge(
        df_visu,
        left_on='INSEE_DEP',
        right_on='code_departement'
    )
    
    
    val_max = carte_data['surrepresentation'].abs().max()
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    carte_data.plot(
        column='surrepresentation',
        cmap='RdBu_r',       # Rouge = surreprésenté, bleu = sous-représenté
        legend=True,
        ax=ax,
        vmin=-val_max,
        vmax=val_max,
        legend_kwds={'label': "(% par rapport\nmoyenne nationale)"}
    )
    ax.set_title(f"Représentation territoriale de {nom_candidat} (Présidentielle 2022)")
    ax.axis('off')
    plt.tight_layout()
    plt.show()

tracer_carte_candidat("LE PEN")